In [2]:
from pathlib import Path
import requests
import pandas as pd
import numpy as np

In [31]:
# Project folders
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Raw data folder:", RAW_DIR)

Project folder: c:\Users\rleue\OneDrive\Desktop\Statistics_ETH\Semester_1\Algorithms_&_Fairness\diabetes_fairness_iml
Raw data folder: c:\Users\rleue\OneDrive\Desktop\Statistics_ETH\Semester_1\Algorithms_&_Fairness\diabetes_fairness_iml\data\raw


In [4]:
BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles"

NHANES_FILES = {
    "demo": "DEMO_L.xpt",   # demographics: age, sex, race/ethnicity, income
    "diq": "DIQ_L.xpt",     # diabetes questionnaire: doctor diagnosis
    "ghb": "GHB_L.xpt",     # glycohemoglobin: HbA1c
    "bmx": "BMX_L.xpt",     # body measures: BMI
    "hiq": "HIQ_L.xpt",     # health insurance/access variables
}

In [5]:
def download_file(filename: str, raw_dir: Path = RAW_DIR) -> Path:
    """
    Downloads one NHANES .xpt file from the CDC if it is not already saved locally.
    """
    url = f"{BASE_URL}/{filename}"
    local_path = raw_dir / filename

    if local_path.exists():
        print(f"Already downloaded: {filename}")
        return local_path

    print(f"Downloading: {filename}")
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    local_path.write_bytes(response.content)
    print(f"Saved to: {local_path}")
    return local_path


file_paths = {
    name: download_file(filename)
    for name, filename in NHANES_FILES.items()
}

Downloading: DEMO_L.xpt
Saved to: c:\Users\rleue\OneDrive\Desktop\Statistics_ETH\Semester_1\Algorithms_&_Fairness\diabetes_fairness_iml\data\raw\DEMO_L.xpt
Downloading: DIQ_L.xpt
Saved to: c:\Users\rleue\OneDrive\Desktop\Statistics_ETH\Semester_1\Algorithms_&_Fairness\diabetes_fairness_iml\data\raw\DIQ_L.xpt
Downloading: GHB_L.xpt
Saved to: c:\Users\rleue\OneDrive\Desktop\Statistics_ETH\Semester_1\Algorithms_&_Fairness\diabetes_fairness_iml\data\raw\GHB_L.xpt
Downloading: BMX_L.xpt
Saved to: c:\Users\rleue\OneDrive\Desktop\Statistics_ETH\Semester_1\Algorithms_&_Fairness\diabetes_fairness_iml\data\raw\BMX_L.xpt
Downloading: HIQ_L.xpt
Saved to: c:\Users\rleue\OneDrive\Desktop\Statistics_ETH\Semester_1\Algorithms_&_Fairness\diabetes_fairness_iml\data\raw\HIQ_L.xpt


In [6]:
def read_xpt(path: Path) -> pd.DataFrame:
    """
    Reads a SAS transport .xpt file into a pandas DataFrame.
    """
    df = pd.read_sas(path, format="xport")
    df.columns = df.columns.str.upper()
    return df


data = {
    name: read_xpt(path)
    for name, path in file_paths.items()
}

for name, df in data.items():
    print(name, df.shape)

demo (11933, 27)
diq (11744, 9)
ghb (7199, 3)
bmx (8860, 22)
hiq (11933, 11)


In [7]:
df = (
    data["demo"]
    .merge(data["diq"], on="SEQN", how="left")
    .merge(data["ghb"], on="SEQN", how="left")
    .merge(data["bmx"], on="SEQN", how="left")
    .merge(data["hiq"], on="SEQN", how="left")
)

print(df.shape)
df.head()

(11933, 68)


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,HIQ011,HIQ032A,HIQ032B,HIQ032C,HIQ032D,HIQ032E,HIQ032F,HIQ032H,HIQ032I,HIQ210
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,...,1.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN,1.0
3,130381.0,12.0,2.0,2.0,5.0,NaN,5.0,7.0,1.0,71.0,...,1.0,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,2.0
4,130382.0,12.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,34.0,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0


In [8]:
# Variables we want for the first version of the project.
# Some health-insurance variables may depend on the exact file/codebook,
# so the code checks whether they exist before selecting them.

variable_map = {
    "SEQN": "id",

    # Demographics
    "RIDAGEYR": "age",
    "RIAGENDR": "sex_code",
    "RIDRETH3": "race_ethnicity_code",
    "INDFMPIR": "income_poverty_ratio",

    # Diabetes target 1: self-reported diagnosis
    "DIQ010": "doctor_diabetes_code",

    # Diabetes target 2: biomarker
    "LBXGH": "hba1c",

    # Body measure
    "BMXBMI": "bmi",

    # Health insurance/access variables
    "HIQ011": "health_insurance_code",
    "HIQ210": "uninsured_past_12mo_code",
}

available_vars = [var for var in variable_map.keys() if var in df.columns]
missing_vars = [var for var in variable_map.keys() if var not in df.columns]

print("Available variables:", available_vars)
print("Missing variables:", missing_vars)

df_project = df[available_vars].rename(columns=variable_map).copy()

df_project.head()

Available variables: ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'INDFMPIR', 'DIQ010', 'LBXGH', 'BMXBMI', 'HIQ011', 'HIQ210']
Missing variables: []


,id,age,sex_code,race_ethnicity_code,income_poverty_ratio,doctor_diabetes_code,hba1c,bmi,health_insurance_code,uninsured_past_12mo_code
0,130378.0,43.0,1.0,6.0,5.00,2.0,5.6,27.0,1.0,2.0
1,130379.0,66.0,1.0,3.0,5.00,2.0,5.6,33.5,1.0,2.0
2,130380.0,44.0,2.0,2.0,1.41,1.0,6.2,29.7,1.0,1.0
3,130381.0,5.0,2.0,7.0,1.53,2.0,NaN,23.8,1.0,2.0
4,130382.0,2.0,1.0,3.0,3.60,2.0,NaN,NaN,1.0,1.0


In [9]:
# Restrict to adults.
# This keeps the analysis clinically simpler and avoids mixing children/adolescents with adults.
df_project = df_project[df_project["age"] >= 18].copy()

# Sex labels
sex_map = {
    1: "Male",
    2: "Female"
}
df_project["sex"] = df_project["sex_code"].map(sex_map)

# Race/ethnicity labels, NHANES RIDRETH3 coding
race_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other/Multi-Racial"
}
df_project["race_ethnicity"] = df_project["race_ethnicity_code"].map(race_map)

# Target 1: self-reported physician-diagnosed diabetes.
# DIQ010: usually 1 = yes, 2 = no, 3 = borderline, 7/9 = refused/don't know.
# For the first analysis, we compare clear yes vs clear no and set other answers to missing.
df_project["diagnosed_diabetes"] = np.select(
    [
        df_project["doctor_diabetes_code"] == 1,
        df_project["doctor_diabetes_code"] == 2,
    ],
    [
        1,
        0,
    ],
    default=np.nan
)

# Target 2: biomarker-based diabetes indicator.
# HbA1c >= 6.5% is commonly used as a diabetes threshold.
# This is not a full medical diagnosis here, only an operational research label.
df_project["hba1c_diabetes"] = np.where(
    df_project["hba1c"].notna(),
    (df_project["hba1c"] >= 6.5).astype(int),
    np.nan
)

# Health insurance recoding, only if variable exists
if "health_insurance_code" in df_project.columns:
    df_project["has_health_insurance"] = np.select(
        [
            df_project["health_insurance_code"] == 1,
            df_project["health_insurance_code"] == 2,
        ],
        [
            1,
            0,
        ],
        default=np.nan
    )

if "uninsured_past_12mo_code" in df_project.columns:
    df_project["uninsured_past_12mo"] = np.select(
        [
            df_project["uninsured_past_12mo_code"] == 1,
            df_project["uninsured_past_12mo_code"] == 2,
        ],
        [
            1,
            0,
        ],
        default=np.nan
    )

df_project.head()

,id,age,sex_code,race_ethnicity_code,income_poverty_ratio,doctor_diabetes_code,hba1c,bmi,health_insurance_code,uninsured_past_12mo_code,sex,race_ethnicity,diagnosed_diabetes,hba1c_diabetes,has_health_insurance,uninsured_past_12mo
0,130378.0,43.0,1.0,6.0,5.00,2.0,5.6,27.0,1.0,2.0,Male,Non-Hispanic Asian,0.0,0.0,1.0,0.0
1,130379.0,66.0,1.0,3.0,5.00,2.0,5.6,33.5,1.0,2.0,Male,Non-Hispanic White,0.0,0.0,1.0,0.0
2,130380.0,44.0,2.0,2.0,1.41,1.0,6.2,29.7,1.0,1.0,Female,Other Hispanic,1.0,0.0,1.0,1.0
6,130384.0,43.0,1.0,1.0,0.63,2.0,NaN,NaN,2.0,NaN,Male,Mexican American,0.0,NaN,0.0,NaN
7,130385.0,65.0,2.0,3.0,5.00,2.0,NaN,NaN,1.0,2.0,Female,Non-Hispanic White,0.0,NaN,1.0,0.0


In [10]:
final_columns = [
    "id",
    "age",
    "sex",
    "race_ethnicity",
    "income_poverty_ratio",
    "bmi",
    "hba1c",
    "diagnosed_diabetes",
    "hba1c_diabetes",
]

# Add access variables if available
optional_columns = [
    "has_health_insurance",
    "uninsured_past_12mo",
]

final_columns = final_columns + [
    col for col in optional_columns if col in df_project.columns
]

df_final = df_project[final_columns].copy()

print(df_final.shape)
df_final.head()

(8153, 11)


,id,age,sex,race_ethnicity,income_poverty_ratio,bmi,hba1c,diagnosed_diabetes,hba1c_diabetes,has_health_insurance,uninsured_past_12mo
0,130378.0,43.0,Male,Non-Hispanic Asian,5.00,27.0,5.6,0.0,0.0,1.0,0.0
1,130379.0,66.0,Male,Non-Hispanic White,5.00,33.5,5.6,0.0,0.0,1.0,0.0
2,130380.0,44.0,Female,Other Hispanic,1.41,29.7,6.2,1.0,0.0,1.0,1.0
6,130384.0,43.0,Male,Mexican American,0.63,NaN,NaN,0.0,NaN,0.0,NaN
7,130385.0,65.0,Female,Non-Hispanic White,5.00,NaN,NaN,0.0,NaN,1.0,0.0


In [11]:
missing_summary = (
    df_final
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_share")
)

missing_summary

,missing_share
hba1c,0.263829
hba1c_diabetes,0.263829
bmi,0.235251
income_poverty_ratio,0.169999
uninsured_past_12mo,0.093708
diagnosed_diabetes,0.033975
has_health_insurance,0.005029
sex,0.000000
age,0.000000
id,0.000000


In [12]:
print("Diagnosed diabetes target:")
print(df_final["diagnosed_diabetes"].value_counts(dropna=False, normalize=True))

print("\nHbA1c diabetes target:")
print(df_final["hba1c_diabetes"].value_counts(dropna=False, normalize=True))

Diagnosed diabetes target:
diagnosed_diabetes
0.0    0.834417
1.0    0.131608
NaN    0.033975
Name: proportion, dtype: float64

HbA1c diabetes target:
hba1c_diabetes
0.0    0.647737
NaN    0.263829
1.0    0.088434
Name: proportion, dtype: float64


In [32]:
output_path = OUTPUT_DIR / "nhanes_diabetes_project_dataframe.csv"
df_final.to_csv(output_path, index=False)

print(f"Saved final dataframe to: {output_path}")

Saved final dataframe to: c:\Users\rleue\OneDrive\Desktop\Statistics_ETH\Semester_1\Algorithms_&_Fairness\diabetes_fairness_iml\data\processed\nhanes_diabetes_project_dataframe.csv


In [14]:
# Variables used as predictors
# Important: hba1c is NOT included as a predictor because it defines hba1c_diabetes.
predictor_cols = [
    "age",
    "sex",
    "race_ethnicity",
    "income_poverty_ratio",
    "bmi",
    "has_health_insurance",
    "uninsured_past_12mo",
]

target_cols = [
    "diagnosed_diabetes",
    "hba1c_diabetes",
]

model_cols = predictor_cols + target_cols

df_model = df_final[model_cols].dropna().copy()

print("Original dataframe shape:", df_final.shape)
print("Model dataframe shape:", df_model.shape)
print("Rows kept:", len(df_model))
print("Share kept:", len(df_model) / len(df_final))

Original dataframe shape: (8153, 11)
Model dataframe shape: (4565, 9)
Rows kept: 4565
Share kept: 0.5599165951183613


In [15]:
print("Diagnosed diabetes target:")
print(df_model["diagnosed_diabetes"].value_counts(normalize=True))
print(df_model["diagnosed_diabetes"].value_counts())

print("\nHbA1c diabetes target:")
print(df_model["hba1c_diabetes"].value_counts(normalize=True))
print(df_model["hba1c_diabetes"].value_counts())

Diagnosed diabetes target:
diagnosed_diabetes
0.0    0.855422
1.0    0.144578
Name: proportion, dtype: float64
diagnosed_diabetes
0.0    3905
1.0     660
Name: count, dtype: int64

HbA1c diabetes target:
hba1c_diabetes
0.0    0.880613
1.0    0.119387
Name: proportion, dtype: float64
hba1c_diabetes
0.0    4020
1.0     545
Name: count, dtype: int64


In [16]:
pd.crosstab(
    df_model["diagnosed_diabetes"],
    df_model["hba1c_diabetes"],
    rownames=["Diagnosed diabetes"],
    colnames=["HbA1c diabetes"],
    margins=True,
    normalize=False
)

HbA1c diabetes,0.0,1.0,All
Diagnosed diabetes,,,
0.0,3817,88,3905
1.0,203,457,660
All,4020,545,4565


In [17]:
pd.crosstab(
    df_model["diagnosed_diabetes"],
    df_model["hba1c_diabetes"],
    rownames=["Diagnosed diabetes"],
    colnames=["HbA1c diabetes"],
    margins=True,
    normalize="index"
)

HbA1c diabetes,0.0,1.0
Diagnosed diabetes,,
0.0,0.977465,0.022535
1.0,0.307576,0.692424
All,0.880613,0.119387


In [18]:
df_model["potential_undiagnosed_diabetes"] = (
    (df_model["diagnosed_diabetes"] == 0) &
    (df_model["hba1c_diabetes"] == 1)
).astype(int)

df_model["potential_undiagnosed_diabetes"].value_counts(normalize=True)

potential_undiagnosed_diabetes
0    0.980723
1    0.019277
Name: proportion, dtype: float64

In [26]:
df_model.groupby("has_health_insurance")["potential_undiagnosed_diabetes"].mean()

has_health_insurance
1.0    0.019277
Name: potential_undiagnosed_diabetes, dtype: float64

In [27]:
df_model.groupby("uninsured_past_12mo")["potential_undiagnosed_diabetes"].mean()

uninsured_past_12mo
0.0    0.019549
1.0    0.013825
Name: potential_undiagnosed_diabetes, dtype: float64

In [29]:
# Rank the income values first so qcut can still create four groups even when many values are tied.
income_rank = df_model["income_poverty_ratio"].rank(method="first")

df_model["income_group"] = pd.qcut(
    income_rank,
    q=4,
    labels=["lowest income", "low-middle income", "high-middle income", "highest income"]
)

df_model.groupby("income_group")["potential_undiagnosed_diabetes"].mean()

income_group
lowest income         0.017513
low-middle income     0.032428
high-middle income    0.017528
highest income        0.009641
Name: potential_undiagnosed_diabetes, dtype: float64

In [33]:
model_output_path = OUTPUT_DIR / "nhanes_diabetes_model_dataframe.csv"
df_model.to_csv(model_output_path, index=False)

print("Saved modelling dataframe to:", model_output_path)

Saved modelling dataframe to: c:\Users\rleue\OneDrive\Desktop\Statistics_ETH\Semester_1\Algorithms_&_Fairness\diabetes_fairness_iml\data\processed\nhanes_diabetes_model_dataframe.csv
